# Lab 5 — Build an OpenAI-Compatible API
**Day 2 Morning | ~45 minutes | Colab CPU**

***

## What You Will Build
By the end of this lab you will have:
1. Written a FastAPI server that exposes `/v1/chat/completions` — the OpenAI wire format
2. Launched it as a background process and exposed it to the internet via an ngrok tunnel
3. Called your own endpoint with the OpenAI Python client — indistinguishable from calling OpenAI directly
4. Added streaming SSE (Server-Sent Events) responses
5. Compared three providers with identical client code (your server, OpenAI direct, Groq)

> **The key idea:** You are *being* the backend today, not calling one.
> Every vLLM, TGI, and Ollama server exposes exactly this shape. Now you know why.

***

## The System You Are Building

```
Your Python code (OpenAI client)
        │  base_url = ngrok public URL
        ▼
YOUR FastAPI server (running on Colab, port 8000)
        │  proxies requests to OpenAI (for now)
        ▼
OpenAI API  ← today's backend
        ↕
In production: swap to vLLM on a GPU server
        └─ same client code, different base_url only
```

Understanding this diagram is the entire lab. Everything else is implementation detail.

------

## The Three Tools You Will Use

| Tool        | What it is                                          | Why we need it                                               |
| ----------- | --------------------------------------------------- | ------------------------------------------------------------ |
| **FastAPI** | Python web framework for building HTTP APIs         | Lets us define the routes (`/health`, `/v1/chat/completions`) and request schemas our server will respond to |
| **uvicorn** | ASGI server (the process that listens on port 8000) | FastAPI defines the routes; uvicorn is the engine that actually receives HTTP requests and hands them to FastAPI |
| **ngrok**   | Tunneling service                                   | Colab runs on a Google server — its port 8000 is not reachable from the internet. ngrok creates a secure public URL that tunnels to port 8000 inside the Colab VM |

Each part has a **Concept** section (read it), a **Code** section (run it), and a **Checkpoint** (confirm you understand it before moving on).

**Coming from Lab 4:** you trained an adapter on a GPU. Day 2 is serving. This notebook does **not** load that adapter — Colab CPU cannot run vLLM. You will write an OpenAI-compatible **proxy** (FastAPI) in front of `gpt-4o-mini`, then point the **same** Lab 1A client at *your* URL. That is the `base_url` swap, now with a server you own.


---

## This Is Not a Demo — The Colab-to-Production Map

A common trap in notebook-based courses is the **demo effect**: something works in Colab and students assume it is a toy that would fail in the real world. This lab is specifically designed to prevent that.

**The server code you write in `server.py` is production code.** The FastAPI patterns, Pydantic schemas, SSE streaming, and `/v1/chat/completions` route are exactly what production LLM serving infrastructure looks like. What changes between Colab and production is not the *code* — it is the *infrastructure wrapper* around it.

| This Lab (Colab) | Why We Use It Here | Production Equivalent |
|---|---|---|
| `subprocess.Popen(uvicorn...)` | A notebook cell blocks if you run a server inside it. `Popen` launches it as a separate process so the notebook continues. | Docker container, systemd service, or Kubernetes pod. The `uvicorn server:app` command inside is **identical**. |
| ngrok tunnel | The Colab VM runs inside Google's data center with no public IP. Nothing on the internet can reach `localhost:8000`. | Nginx or Apache reverse proxy, AWS Application Load Balancer, or Cloudflare Tunnel — all do the same job: forward public HTTPS traffic to the local port. |
| OpenAI as the backend model | No GPU on a free Colab CPU runtime. OpenAI gives us a real model response without needing local hardware. | vLLM or TGI running on a GPU instance (EC2 `g5.xlarge`, Lambda Labs A10, Hetzner GPU box). One env var swap: `BACKEND_BASE_URL`. |
| `time.sleep(3)` waiting for startup | Simple: just wait 3 seconds before checking. | Docker `HEALTHCHECK` directive or Kubernetes readiness probe. Same concept — don't route traffic until the server is ready. |
| Colab VM | Free notebook runtime. No persistent disk, no public IP, recycles after idle. | Any cloud instance with a real IP: EC2, GCP Compute Engine, Lambda Labs, Vast.ai. |

**Exactly what is identical between today's lab and a production deployment:**

```
STAYS EXACTLY THE SAME                    CHANGES (infrastructure wrapper only)
──────────────────────────────────────    ──────────────────────────────────────────
server.py — all of it                     How server launches: subprocess → Docker
/v1/chat/completions route + schema       Where it runs: Colab → cloud GPU instance
FastAPI + uvicorn                         How it's reachable: ngrok → real domain + TLS
Pydantic request validation               What serves the model: OpenAI proxy → vLLM
SSE streaming protocol (data: ...\n\n)   Process management: manual → systemd / k8s
OpenAI Python client call pattern         Auth: none → real API key middleware
The one-line backend swap (base_url)      Cost: free tier → ~$1–3/hr GPU instance
```

**The two-step production upgrade:**

```
Step 1 — Give the server a real address:
  Colab + ngrok  →  EC2 g5.xlarge (public IP) + Nginx on port 443 + TLS cert

Step 2 — Replace the OpenAI proxy with a local model (one env var):
  BACKEND_BASE_URL = "https://api.openai.com/v1"    # today
  BACKEND_BASE_URL = "http://localhost:8000/v1"      # production, vLLM on same machine
```

The vLLM command that makes Step 2 work (runs on the GPU server — same port, same routes as our server):
```bash
python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen2.5-7B-Instruct \
  --quantization awq \
  --host 0.0.0.0 --port 8000
# ↑ Drop-in replacement. Your client code doesn't change. Only base_url does.
```

> **The single most important idea in this lab:** The OpenAI wire format (`/v1/chat/completions`, JSON in, JSON out) is an open standard, not a vendor product. Every major open-source LLM serving engine — vLLM, Ollama, TGI, LiteLLM — implements it. That means your application code is permanently decoupled from your infrastructure. You swap providers by changing one string.

---

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} fastapi uvicorn pyngrok openai httpx python-dotenv

### 📦 What Was Just Installed

- **fastapi** — the web framework. You define routes and schemas in Python; FastAPI turns them into an HTTP API with automatic validation and documentation.
- **uvicorn** — the ASGI server. FastAPI is just a Python object; uvicorn is the process that binds to a port, listens for HTTP requests, and calls your FastAPI code.
- **pyngrok** — the Python client for ngrok. It opens a public tunnel from a URL on ngrok's servers to a local port inside this runtime.
- **openai** — the OpenAI Python SDK. We use it both to *call* OpenAI (from our server) and to *call our own server* (as a client), proving they are interchangeable.
- **httpx** — an HTTP client. Used for health checks.

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secrets you added
    os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
    os.environ["NGROK_AUTH_TOKEN"] = userdata.get("NGROK_AUTH_TOKEN")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"),   "Add OPENAI_API_KEY as a Colab Secret or to .env"
assert os.environ.get("NGROK_AUTH_TOKEN"), "Add NGROK_AUTH_TOKEN as a Colab Secret or to .env"

OPENAI_API_KEY   = os.environ["OPENAI_API_KEY"]
NGROK_AUTH_TOKEN = os.environ["NGROK_AUTH_TOKEN"]
OPENAI_BASE_URL  = "https://api.openai.com/v1"
DEFAULT_MODEL    = "gpt-4o-mini"
print(f"Ready — {DEFAULT_MODEL}, ngrok token found")

***

## 🧠 Concepts Before Code — What Is FastAPI?

**FastAPI** is a modern Python framework for building HTTP APIs. You write Python functions and decorate them with route definitions — FastAPI handles parsing the incoming HTTP request, validating it against a schema you define, calling your function, and serialising the response back to JSON.

A minimal example of what we are about to write:

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):   # ← Pydantic schema: defines the expected JSON shape
    model: str
    messages: list

@app.post("/v1/chat/completions")  # ← route decorator: this function runs when POST /v1/chat/completions arrives
def chat(req: ChatRequest):
    return {"choices": [...]}
```

Three things to understand:
- **`app = FastAPI()`** creates the application object — the thing uvicorn will run
- **`BaseModel` (Pydantic)** defines the shape of the JSON body. If the request is missing a required field, FastAPI automatically returns a 422 error before your code even runs.
- **`@app.post("/v1/chat/completions")`** registers the function as the handler for `POST` requests to that URL path. FastAPI auto-generates a `/docs` page from these decorators.

**The OpenAI wire format** is just an HTTP API with a specific JSON shape. Our server implements the same shape — which is why the OpenAI Python client can call it without any changes.


### How to Read the Server Cells

`server.py` is written in **four short cells**. The first creates the file (`%%writefile server.py`); the next three append to it (`%%writefile -a`). Run them top to bottom once. If you edit one, re-run all four from A1.1.

| Cell | What it adds | Why it matters |
|---|---|---|
| A1.1 | app object + backend config from env vars | Swap the backend by changing env vars, not code |
| A1.2 | Pydantic request schemas | Wrong JSON is rejected before your code runs |
| A1.3 | `/health`, `/v1/models` | What load balancers and clients probe |
| A1.4 | `/v1/chat/completions` with streaming | The OpenAI wire format, including SSE |

> **The deployment lesson:** today the server forwards to OpenAI. In production, point `BACKEND_BASE_URL` at vLLM and nothing else changes.

***

## Part A — Write and Launch the Server (20 min)

We write `server.py` from a notebook cell, then launch it as a background process and expose it to the internet.

### Why write a file from inside a notebook?
A web server is a long-running process — it must stay alive while the notebook continues executing. You cannot run a long-lived process inside a notebook code cell (the cell would block forever). The pattern is:
1. Write the server code to a `.py` file from inside the notebook
2. Launch it as a separate background process using `subprocess.Popen`
3. The notebook kernel continues; the server process runs in parallel

### Why a background subprocess?
`subprocess.Popen(...)` starts a child process and returns immediately — the notebook does not wait for it to finish. The server keeps running until you explicitly terminate it (the cleanup cell at the end of the lab). If you re-run Cell A2 without running cleanup first, you will get a "port 8000 already in use" error. If that happens, run the cleanup cell and re-run A2.

### The production equivalent of this step
The `uvicorn server:app --host 0.0.0.0 --port 8000` command you are about to run via `subprocess.Popen` is **word-for-word identical** to what you would put inside a `Dockerfile`:

```dockerfile
# Production Dockerfile — the uvicorn command is identical
FROM python:3.11-slim
COPY server.py .
RUN pip install fastapi uvicorn openai
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"]
```

What changes: how the process is managed (subprocess → Docker/systemd/k8s). What does not change: the command itself, `server.py`, the port, and the routes.

### A1.1 — App object and backend config

In [ ]:
%%writefile server.py
import os, json, time, uuid
from typing import List, Optional
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from openai import OpenAI

app = FastAPI(title="My LLM API", version="0.1.0")

# Which backend this server forwards to. Change the env vars and restart; nothing else changes.
BACKEND_API_KEY  = os.environ.get("BACKEND_API_KEY", "")
BACKEND_BASE_URL = os.environ.get("BACKEND_BASE_URL", "https://api.openai.com/v1")
DEFAULT_MODEL    = os.environ.get("DEFAULT_MODEL", "gpt-4o-mini")

backend = OpenAI(api_key=BACKEND_API_KEY, base_url=BACKEND_BASE_URL)

### A1.2 — Request schemas

Pydantic models define the JSON the server accepts. If a caller sends the wrong shape, FastAPI returns `422` before your function runs. These two classes *are* the OpenAI request format. `-a` appends to the file.

In [ ]:
%%writefile -a server.py

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = DEFAULT_MODEL
    messages: List[Message]
    stream: bool = False
    temperature: float = 0.7
    max_tokens: Optional[int] = 500

### A1.3 — Two small routes

`/health` is what a load balancer polls. `/v1/models` is what an OpenAI client can list. Each is a plain function with a decorator.

In [ ]:
%%writefile -a server.py

@app.get("/health")
def health():
    return {"status": "ok", "backend": BACKEND_BASE_URL, "model": DEFAULT_MODEL}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": DEFAULT_MODEL, "object": "model", "owned_by": "custom-server"}]}

### A1.4 — The main route

`POST /v1/chat/completions` has two paths. **Non-streaming:** forward to the backend, then reshape the reply into the OpenAI response format. **Streaming:** `generate()` yields one Server-Sent-Events line per token delta (`data: {...}\n\n`) and finishes with `data: [DONE]`. That is why ChatGPT looks like it is typing.

In [ ]:
%%writefile -a server.py

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    msgs = [{"role": m.role, "content": m.content} for m in req.messages]

    if not req.stream:
        resp = backend.chat.completions.create(
            model=req.model, messages=msgs, temperature=req.temperature, max_tokens=req.max_tokens
        )
        return {
            "id": f"chatcmpl-{uuid.uuid4().hex[:8]}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": req.model,
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": resp.choices[0].message.content}}],
            "usage": resp.usage.model_dump(),
        }

    def generate():
        stream = backend.chat.completions.create(
            model=req.model, messages=msgs, temperature=req.temperature, stream=True
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                data = {"id": f"chatcmpl-{uuid.uuid4().hex[:8]}", "object": "chat.completion.chunk",
                        "created": int(time.time()), "model": req.model,
                        "choices": [{"index": 0, "delta": {"content": delta}, "finish_reason": None}]}
                yield f"data: {json.dumps(data)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(generate(), media_type="text/event-stream")

**Checkpoint:** four cells, one file. Print it to see the whole server in one place — this is what production would copy into a container.

In [ ]:
print(open("server.py").read())

### Step A2 — Launch the server

This cell starts uvicorn in the background on `127.0.0.1:8000` and polls `GET /health` until it answers. Logs go to `uvicorn.log` (never a stdout pipe — a full pipe freezes the server on Colab). If health never comes up, open `uvicorn.log`.

Step A3 then opens an ngrok tunnel with `pyngrok` (still [ngrok's official Colab path](https://ngrok.com/docs/using-ngrok-with/googleColab)). You need a free account and the `NGROK_AUTH_TOKEN` secret. Gradio (Lab 7) uses its own `share=True` tunnel instead.

**Colab / free-plan quirks (read once):**

| What you will see | What to do |
|---|---|
| Browser interstitial “Visit Site” on the ngrok URL | Click **Visit Site**. That warning is for *browsers*. Our Python client sends `ngrok-skip-browser-warning` so API calls skip it. |
| `ERR_NGROK_107` / invalid authtoken | Check the `NGROK_AUTH_TOKEN` secret. Copy the token from the ngrok dashboard again. |
| Health check on localhost fails | Port 8000 did not come up. Open `uvicorn.log`, or run the cleanup cell and retry A2. |
| Tunnel fails but localhost health is OK | Keep going: B1–B3 use `http://127.0.0.1:8000`. You still built an OpenAI-compatible server. |
| Port 8000 already in use | Cleanup cell, then A2 again. |

After a successful tunnel: open **Swagger** (`/docs`) on your phone. That is the “this is a real API” moment.

In [ ]:
import subprocess, time, httpx, sys

LOCAL_URL = "http://127.0.0.1:8000"
env = {**os.environ, "BACKEND_API_KEY": OPENAI_API_KEY, "BACKEND_BASE_URL": OPENAI_BASE_URL, "DEFAULT_MODEL": DEFAULT_MODEL}

# Logs go to a file. Never PIPE stdout on Colab: a full pipe freezes uvicorn.
log = open("uvicorn.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT, env=env,
)

healthy = False
for _ in range(20):                       # up to 10 seconds
    try:
        httpx.get(f"{LOCAL_URL}/health", timeout=1).raise_for_status()
        healthy = True
        break
    except Exception:
        time.sleep(0.5)

assert healthy, "Server did not start. Open uvicorn.log for the reason."
print("Local health OK —", LOCAL_URL)

**Checkpoint:** `Local health OK`. The server is alive inside this runtime, but nothing outside can reach port 8000 yet.

### Step A3 — Open the public tunnel

ngrok gives the local port an HTTPS URL. If the tunnel fails (bad token, network policy), the lab continues on localhost — the OpenAI-compatible server is the lesson, the public URL is the demo.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
NGROK_HEADERS = {"ngrok-skip-browser-warning": "true"}   # free plan shows a browser warning page; API calls skip it

try:
    SERVER_URL = ngrok.connect(8000).public_url
    print("Public URL :", SERVER_URL)
    print("Swagger UI :", f"{SERVER_URL}/docs")
    print("If a browser shows an ngrok warning page, click Visit Site.")
except Exception as e:
    SERVER_URL = LOCAL_URL
    print("ngrok tunnel did not open:", e)
    print("Continuing on", SERVER_URL)

#### ✅ Part A Checkpoint
- [ ] You have written `server.py` to disk.
- [ ] You have launched it via `uvicorn`.
- [ ] You can click the Swagger UI link above and see the endpoints.


***

## Part B — Call Your Server (15 min)

Your server is live and has a public URL. This part proves it speaks the OpenAI protocol correctly — meaning any client that works with OpenAI will work with your server without modification.

**What you will test, cell by cell:**
- **B1 (Health check):** A raw HTTP GET to `/health` — confirms the server process is up and responding
- **B2 (OpenAI client → your server):** The OpenAI Python SDK pointed at your ngrok URL. The client code is *identical* to calling OpenAI directly — only `base_url` changed.
- **B3 (Streaming):** The same client with `stream=True` — tokens print as they arrive, one delta at a time
- **B4 (Provider comparison):** The same client code calling three different backends. Watch: only `base_url` and `api_key` change. The call, the response shape, and the output handling are identical.
- **B5 (vs vLLM):** What our server cannot do at scale, and what vLLM adds

> ⚠️ **Remember to run the cleanup cell at the bottom when you finish.** Leaving the server and ngrok tunnel running wastes Colab resources and your ngrok connection.

If ngrok failed, `SERVER_URL` is `http://127.0.0.1:8000`. That is enough to prove the protocol. The public URL is how a partner or your phone reaches the same server.


In [ ]:
import httpx

def get_json(url: str):
    headers = NGROK_HEADERS if "ngrok" in url else None
    r = httpx.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    return r.json()

print("Local  :", get_json(f"{LOCAL_URL}/health"))
if SERVER_URL != LOCAL_URL:
    print("Public :", get_json(f"{SERVER_URL}/health"))
else:
    print("Public : (no tunnel — using localhost)")


### Step B2 — The OpenAI Client Calling Your Server

This is the payoff of the lab. The code below is the standard OpenAI SDK — the exact same code you would use to call `api.openai.com`. The only difference is `base_url=f'{SERVER_URL}/v1'`.

```python
my_client = OpenAI(base_url=f'{SERVER_URL}/v1', api_key='not-needed')
```

The `api_key='not-needed'` works because our server does not validate incoming keys — it uses its own key to call OpenAI on your behalf. In production you would add auth middleware.

**What this demonstrates:** Any application already using the OpenAI SDK can point at your server, a vLLM server, an Ollama server, or any other OpenAI-compatible backend by changing one line. This is the "OpenAI-compatible" contract that the entire open-source LLM serving ecosystem is built on.

> **Instructor note:** This is the payoff. The client does not know which server it is talking to.


In [ ]:
from openai import OpenAI

def make_client(base_url: str, api_key: str = "not-needed"):
    """Same OpenAI SDK. Extra header only when talking through free ngrok."""
    kwargs = {"base_url": base_url, "api_key": api_key}
    if "ngrok" in base_url:
        kwargs["default_headers"] = NGROK_HEADERS
    return OpenAI(**kwargs)

my_client = make_client(f"{SERVER_URL}/v1")

resp = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {"role": "system", "content": "You are an LLM deployment expert."},
        {"role": "user", "content": "What are 3 advantages of vLLM over a naive FastAPI server?"},
    ],
)
print(resp.choices[0].message.content)
print()
print("Tokens:", resp.usage)


### Step B3 — Streaming (Server-Sent Events)

Without streaming, the server generates the full response, then sends it all at once. With streaming, the server sends each token *as it is generated* using **Server-Sent Events (SSE)** — an HTTP standard where the server sends a stream of `data: {...}\n\n` lines, and the connection stays open until a `data: [DONE]` sentinel arrives.

**In our server (`server.py`), the streaming path:**
1. Calls `backend.chat.completions.create(..., stream=True)` — OpenAI sends back token chunks
2. For each chunk, yields a JSON line: `data: {"choices": [{"delta": {"content": "..."}}]}\n\n`
3. After the last token, yields `data: [DONE]\n\n`

**In the client below:**
- `stream=True` tells the SDK to return an iterator instead of waiting for the full response
- `chunk.choices[0].delta.content` is the token delta for that chunk
- `print(delta, end='', flush=True)` prints each token immediately without a newline — this is what produces the "typing" effect

This is the same mechanism behind every streaming LLM interface you have used.


In [ ]:
# Cell B3 — Streaming from your server
print('Streaming from our server:')
print('─' * 60)
stream = my_client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{'role': 'user', 'content': 'List 5 things that can go wrong when serving LLMs. Be brief.'}],
    stream=True
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print('\n' + '─' * 60)

### Step B4 — Provider Comparison: Three Backends, One Client

This cell runs the same question against three backends using identical client code. The only thing that changes between each call is `base_url` and `api_key`.

```
Provider 1: base_url = your ngrok URL    → your server → proxies to OpenAI
Provider 2: base_url = api.openai.com    → OpenAI directly (gpt-4o-mini)
Provider 3: base_url = api.openai.com    → OpenAI directly (gpt-4o)
```

**Why does this matter?** This is the architecture of the entire LLM serving ecosystem. Tools like LiteLLM, OpenRouter, and Helicone are variations of this pattern — they all expose the OpenAI-compatible shape so that application code never needs to change when you swap providers, models, or infrastructure.

Watch the output: same question, same client code, three different answers. The differences come from model quality, not from anything in your application.

> **Instructor note:** Change only base_url + api_key. The rest is identical.


In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # not on Colab, or no such secret
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

providers = {
    "Our FastAPI → OpenAI":        {"base_url": f"{SERVER_URL}/v1", "api_key": "not-needed", "model": DEFAULT_MODEL},
    "OpenAI direct (gpt-4o-mini)": {"base_url": OPENAI_BASE_URL, "api_key": OPENAI_API_KEY, "model": DEFAULT_MODEL},
}
if GROQ_API_KEY:
    providers["Groq llama-3.1-8b-instant"] = {
        "base_url": "https://api.groq.com/openai/v1", "api_key": GROQ_API_KEY, "model": "llama-3.1-8b-instant"}
else:
    print("No GROQ_API_KEY — comparing your server vs OpenAI only.\n")

question = "In one sentence: what is PagedAttention?"
for name, cfg in providers.items():
    c = make_client(cfg["base_url"], cfg["api_key"])
    r = c.chat.completions.create(model=cfg["model"], messages=[{"role": "user", "content": question}])
    print(f"[{name}]\n  {r.choices[0].message.content}\n")

## Part C — Understanding vLLM

Our FastAPI server works perfectly — but it has a fundamental limitation: it handles one request at a time. If two users hit `/v1/chat/completions` simultaneously, the second one waits.

**vLLM** is a production LLM inference engine (from the UC Berkeley Sky Lab) built to solve exactly this. It exposes the same OpenAI-compatible `/v1/chat/completions` endpoint — so it is a drop-in replacement for our server — but adds three things we don't have:

1. **Continuous batching:** vLLM processes multiple requests simultaneously by interleaving their token generation steps. Our server generates tokens for one request from start to finish before starting the next.

2. **PagedAttention:** During generation, the model needs to store key/value tensors for every token in the prompt (the KV cache). Without memory management, this wastes GPU RAM because requests have unpredictable lengths. PagedAttention manages the KV cache like virtual memory pages — non-contiguous, allocated on demand — resulting in 20–30x throughput improvement over naive serving.

3. **Quantization support (AWQ, GPTQ):** vLLM has native support for running quantized models, reducing GPU memory requirements without significant quality loss.

The cell below prints a side-by-side comparison. After reading it, you should be able to answer: *"When would you use our server vs vLLM?"*


In [ ]:
# Cell B5 — What vLLM adds on top of what we built
print('''
OUR FASTAPI SERVER (built today)     vs     vLLM
─────────────────────────────────────────────────────────────────
✅ OpenAI-compatible /v1/chat/completions   ✅ Same
✅ Streaming SSE                             ✅ Same
✅ Easy to read and modify                   ❌ More complex
❌ Sequential: one request at a time         ✅ Continuous batching: N concurrent users
❌ No KV-cache management                   ✅ PagedAttention: non-contiguous KV pages
❌ No GPU memory optimisation               ✅ 20–30x throughput vs naive serving
❌ No quantisation support built-in         ✅ AWQ / GPTQ / bitsandbytes native

When to use ours: development, testing, internal tools with low traffic
When to use vLLM: production, real user traffic, GPU cost-sensitive deployments

Production vLLM command (run on a GPU server, not Colab):
  python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct \\
    --quantization awq \\
    --host 0.0.0.0 --port 8000
  → Drop-in replacement for our server. Same client code.
''')

#### ✅ Part B Checkpoint

Confirm you can answer these before moving to cleanup:
- [ ] What is the only line that changed between calling OpenAI directly and calling your server?
- [ ] What does `stream=True` change about how the server sends its response?
- [ ] Why can our server not handle 100 simultaneous users well?
- [ ] What would you change in Cell A2 to point the server at a vLLM instance instead of OpenAI?

> **Answer to the last question:** Change the `BACKEND_BASE_URL` environment variable to your vLLM server's address (e.g. `http://your-gpu-server:8000/v1`) and `BACKEND_API_KEY` to any non-empty string (vLLM doesn't require a real key by default). Every line of client code in Part B stays exactly the same.

***

⚠️ **Run the cleanup cell below now.** It stops the uvicorn process and closes the ngrok tunnel. If you skip this and close the notebook, both will remain as zombie processes until Colab recycles the VM.


In [ ]:
server_proc.terminate()
try:
    ngrok.kill()
except Exception:
    pass
print("Server and ngrok agent stopped.")


***

## ✅ Lab 5 Complete

You should now have:
- [ ] `server.py` written and launched via uvicorn subprocess
- [ ] Public Swagger UI URL opened and tested in a browser
- [ ] OpenAI Python client successfully calling your own endpoint (B2)
- [ ] Streaming printed token-by-token (B3)
- [ ] Three-provider comparison from identical client code (B4)
- [ ] vLLM comparison read and understood — you know when to use each (B5)
- [ ] Cleanup cell run — server and tunnel are stopped

## 🧠 Key Takeaways

1. **An LLM API is just HTTP** — a `POST` endpoint that accepts JSON and returns JSON. Any web framework can implement it.
2. **The OpenAI wire format is a standard** — the entire open-source serving ecosystem (vLLM, Ollama, TGI, LiteLLM) implements it so your client code never changes.
3. **Streaming = SSE** — the server holds the connection open and sends token deltas as `data: {...}\n\n` lines. This is not special to LLMs; it is a standard HTTP pattern.
4. **Naive serving doesn't scale** — sequential request handling works for demos; production needs continuous batching (vLLM).
5. **ngrok = localhost for demos** — in production this becomes a real domain + load balancer + TLS cert, but the pattern is identical.

## Stretch Goals

1. **Request logging:** Add a `@app.middleware('http')` that logs timestamp, prompt character count, and latency for every request.
2. **API key auth:** Add a check in `chat_completions` that returns HTTP 401 if an `Authorization: Bearer` header is missing or wrong.
3. **Metrics endpoint:** Add `GET /metrics` that returns total request count, average latency, and last error message.
4. **Swap the backend to Groq:** Change `BACKEND_BASE_URL` to `https://api.groq.com/openai/v1` and `BACKEND_API_KEY` to your Groq key. All client code stays the same.
5. **LiteLLM in one command:** Run `uv pip install litellm` then `litellm --model openai/gpt-4o-mini`. It creates a ready-made OpenAI-compatible proxy in one CLI command — the industrial version of what you built today.

## Next

[Lab 6 — RAG Pipeline](../06_RAG_Pipeline/README.md) — still CPU. You retrieve course text with MiniLM + Chroma (Lab 0's vector store, grown up), then generate with `gpt-4o-mini`. The FastAPI server is optional behind that generator; the notebook calls OpenAI directly so the RAG lesson stays visible.
